# Train a custom YOLO model on Colab (free GPU)

Works for any Roboflow **YOLOv11** dataset zip: gloves + shoes, pen, button, cap, mask, beard.
Run the cells **top to bottom** (Runtime > Run all also works after you fill in cell 3).

**Before you start**
1. Runtime > Change runtime type > **T4 GPU** > Save.
2. Upload your dataset zip to Google Drive (e.g. `My Drive/Gloves-Shoes.v1i.yolov11.zip`), or use the upload option in cell 3 for small zips.
3. Fill in the settings in **cell 3**.

**When it finishes** the notebook downloads one `.pt` file. Copy it to `D:\priya\Live CCTV analytics\models\`.

In [ ]:
# 1) GPU check
import shutil, subprocess
if shutil.which("nvidia-smi") is None:
    raise SystemExit("NO GPU FOUND -> Runtime > Change runtime type > T4 GPU > Save, then Runtime > Restart session and run again")
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)

In [ ]:
# 2) Install the same Ultralytics version used on your PC (so the model loads there)
!pip -q install "ultralytics==8.4.176"
import ultralytics; print("ultralytics", ultralytics.__version__)

In [ ]:
# 3) SETTINGS  <-- edit this cell
DATASET_ZIP = "/content/drive/MyDrive/PEN.v2i.yolov11.zip"   # path of the zip on Google Drive
USE_DRIVE   = True      # True: read the zip from Google Drive.  False: you will be asked to upload it (small zips only)
OUTPUT_NAME = "pen_v3.pt"   # file name you will download. gloves+shoes: gloves_shoes.pt | pen: pen.pt | button: button.pt

EPOCHS = 60             # 40-80 is typical on GPU. Training stops early if it stops improving (patience).
IMGSZ  = 960            # use 640 for small things (pens, buttons, gloves)
BATCH  = 16
FRACTION = 1.0          # 1.0 = use all training images

# The dataset's own class names are used (e.g. barefeet, gloves, no_gloves, shoes).
# If you want different names, e.g. {"barefeet": "no_shoes"}, leave this empty and use `rename:` in config.yaml instead.
CLASS_NAMES = ['pen']      # None = use the dataset's own names. For the pen dataset use ["pen"] if the name in the dataset is different (e.g. "Pen")
DROP_LEAKED_VALID = True   # remove validation images that are augmented copies of training images (gives an honest score)

In [ ]:
# 4) Get the zip
import os, shutil
ZIP_LOCAL = "/content/dataset.zip"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    assert os.path.exists(DATASET_ZIP), f"Zip not found: {DATASET_ZIP}  (check the name/path in cell 3)"
    shutil.copy(DATASET_ZIP, ZIP_LOCAL)
else:
    from google.colab import files
    up = files.upload()
    shutil.move(next(iter(up)), ZIP_LOCAL)
print("zip size (MB):", round(os.path.getsize(ZIP_LOCAL) / 1e6, 1))

In [ ]:
# 5) Unzip, find the dataset, write data.yaml, remove leaked validation images
import zipfile, glob, yaml, collections
from pathlib import Path

DS = Path("/content/dataset")
if DS.exists():
    shutil.rmtree(DS)
with zipfile.ZipFile(ZIP_LOCAL) as z:
    bad = [n for n in z.namelist() if n.startswith("/") or ".." in Path(n).parts]
    assert not bad, f"unsafe paths in zip: {bad[:3]}"
    z.extractall(DS)

# the dataset root is the folder that contains train/images
roots = [p.parent.parent for p in DS.rglob("images") if p.parent.name == "train"]
assert roots, "Could not find a train/images folder inside the zip"
ROOT = roots[0]
VAL = "valid" if (ROOT / "valid" / "images").exists() else "val"
print("dataset root:", ROOT, "| validation folder:", VAL)

src = yaml.safe_load((ROOT / "data.yaml").read_text())
names = src["names"]
names = [names[i] for i in sorted(names)] if isinstance(names, dict) else list(names)
if CLASS_NAMES:
    assert len(CLASS_NAMES) == len(names), f"CLASS_NAMES has {len(CLASS_NAMES)} names but the dataset has {len(names)} classes"
    names = list(CLASS_NAMES)
print("classes:", dict(enumerate(names)))

def base(p):  # Roboflow adds ".rf.<hash>" to augmented copies of the same source photo
    return Path(p).name.split(".rf.")[0]

tr_bases = {base(p) for p in glob.glob(str(ROOT / "train" / "images" / "*"))}
if DROP_LEAKED_VALID:
    removed = 0
    for img in glob.glob(str(ROOT / VAL / "images" / "*")):
        if base(img) in tr_bases:
            os.remove(img)
            lab = ROOT / VAL / "labels" / (Path(img).stem + ".txt")
            if lab.exists():
                lab.unlink()
            removed += 1
    print("validation images removed because the same photo is in training:", removed)

def stats(split):
    imgs = len(glob.glob(str(ROOT / split / "images" / "*")))
    per = collections.Counter()
    for f in glob.glob(str(ROOT / split / "labels" / "*.txt")):
        for line in open(f):
            t = line.split()
            if t:
                per[names[int(t[0])]] += 1
    return imgs, dict(per)

for s in ("train", VAL):
    n, per = stats(s)
    print(f"{s:6s}: {n} images, labelled objects per class: {per}")

DATA_YAML = "/content/data.yaml"
Path(DATA_YAML).write_text(yaml.safe_dump({
    "path": str(ROOT), "train": "train/images", "val": f"{VAL}/images",
    "names": {i: n for i, n in enumerate(names)},
}))

In [ ]:
# 6) Train (about 30-60 min on a T4 for ~2000 images)
from ultralytics import YOLO
model = YOLO("yolo11n.pt")   # small model: fast enough for your CPU PC at run time
results = model.train(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, fraction=FRACTION,
    device=0, workers=2, patience=20, project="/content/runs", name="train", exist_ok=True,
)
BEST = Path(results.save_dir) / "weights" / "best.pt"
print("best weights:", BEST)

In [ ]:
# 7) Score per class  (look at every class, not just the "all" row)
m = YOLO(str(BEST))
metrics = m.val(data=DATA_YAML, imgsz=IMGSZ, device=0, plots=False)
print()
print(f"{'class':15s} {'mAP50':>7s}")
for idx, ap in zip(metrics.box.ap_class_index, metrics.box.ap50):
    print(f"{names[int(idx)]:15s} {ap:7.3f}")
print(f"{'ALL':15s} {metrics.box.map50:7.3f}   (precision {metrics.box.mp:.3f}, recall {metrics.box.mr:.3f})")

In [ ]:
# 8) Look at a few predictions on validation photos
import matplotlib.pyplot as plt, cv2, random
imgs = glob.glob(str(ROOT / VAL / "images" / "*"))
random.seed(1)
pick = random.sample(imgs, min(6, len(imgs)))
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, p in zip(axes.ravel(), pick):
    r = m.predict(p, imgsz=IMGSZ, conf=0.35, device=0, verbose=False)[0]
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# 9) Save + download
OUT = Path("/content") / OUTPUT_NAME
shutil.copy(BEST, OUT)
if USE_DRIVE:
    shutil.copy(OUT, Path("/content/drive/MyDrive") / OUTPUT_NAME)   # backup copy in your Drive
    print("backup saved in Google Drive:", OUTPUT_NAME)
from google.colab import files
files.download(str(OUT))

## After the download (on your PC)

1. Copy the `.pt` file into `D:\priya\Live CCTV analytics\models\`.
2. Check its class names:  `python tools\inspect_model.py models\gloves_shoes.pt`
3. Test it on the webcam:  `python detectors\gloves_detection.py`  or  `python detectors\shoes_detection.py`

**Gloves + shoes model:** `config.yaml` already uses `models\gloves_shoes.pt` for both detectors.
**Pen model:** use `OUTPUT_NAME = "pen.pt"` and the pen dataset zip.

If a class scores low in cell 7 (below about 0.5), more photos of that class from your own camera will help more than longer training.